# Multilingual GoEmotions (Expansion)

The purpose of this project is to expand the [Multilingual GoEmotions](https://huggingface.co/datasets/AnasAlokla/multilingual_go_emotions) dataset with more supported languages in a programmatic way by generating synthetic translations

**Already supported languages**
- English (Original Dataset) -> EN
- Arabic -> AR
- French -> FR
- Spanish -> SP
- German -> GR
- Turkish -> TR

**New supported languages**
- Italian -> IT

---

0. Load ENVs & Prepare Enviroment

In [1]:
import os
from dotenv import load_dotenv

load_dotenv(".env");
GOOGLE_GENAI_APIKEY = os.getenv("GOOGLE_GENAI_APIKEY")

cacheDir = "./cache"
if (not os.path.isdir(cacheDir)):
    os.mkdir(cacheDir)

1. Download, store and retrive multilingual_go_emotions

In [2]:
from datasets import load_dataset, load_from_disk

if (os.path.isdir("./data/multilingual_go_emotions")):
    # Retrive
    dataset_goemotions = load_from_disk("./data/multilingual_go_emotions")
else:
    # Download, store
    dataset_goemotions = load_dataset("AnasAlokla/multilingual_go_emotions")
    dataset_goemotions.save_to_disk("./data/multilingual_go_emotions")

dataset_original_train = dataset_goemotions["train"]
dataset_original_test = dataset_goemotions["test"]
dataset_original_validation = dataset_goemotions["validation"]

2. Extract EN samples from datasets

In [3]:
dataset_goemotions_train = dataset_goemotions["train"].filter(lambda x: x["language"] == "en")
dataset_goemotions_test = dataset_goemotions["test"].filter(lambda x: x["language"] == "en")
dataset_goemotions_validation = dataset_goemotions["validation"].filter(lambda x: x["language"] == "en")

3. Assign Numerical IDs to every sample and divide datasets into batches of 100 samples

In [4]:
dataset_goemotions_train = dataset_goemotions_train.map(lambda ex, i: {"serial": f"id_{i+1:06d}"}, with_indices=True).batch(batch_size=100)
dataset_goemotions_test = dataset_goemotions_test.map(lambda ex, i: {"serial": f"id_{i+1:06d}"}, with_indices=True).batch(batch_size=100)
dataset_goemotions_validation = dataset_goemotions_validation.map(lambda ex, i: {"serial": f"id_{i+1:06d}"}, with_indices=True).batch(batch_size=100)

4. Prepare Gemini for inference

In [5]:
from google import genai
from google.genai import types

def generate(prompt, system_prompt):
    api = genai.Client(api_key=GOOGLE_GENAI_APIKEY)
    return api.models.generate_content(
        model="gemini-3.1-flash-lite-preview",
        contents=[
            types.Content(
                role="user",
                parts=[
                    types.Part.from_text(text=prompt),
                ],
            )
        ],
        config=types.GenerateContentConfig(
            temperature=0.3,
            thinking_config=types.ThinkingConfig(
                thinking_level="HIGH",
            ),
            safety_settings=[
                types.SafetySetting(
                    category="HARM_CATEGORY_HARASSMENT",
                    threshold="BLOCK_NONE",  # Block none
                ),
                types.SafetySetting(
                    category="HARM_CATEGORY_HATE_SPEECH",
                    threshold="BLOCK_NONE",  # Block none
                ),
                types.SafetySetting(
                    category="HARM_CATEGORY_SEXUALLY_EXPLICIT",
                    threshold="BLOCK_NONE",  # Block none
                ),
                types.SafetySetting(
                    category="HARM_CATEGORY_DANGEROUS_CONTENT",
                    threshold="BLOCK_NONE",  # Block none
                ),
            ],
            response_mime_type="application/json",
            system_instruction=[
                types.Part.from_text(text=system_prompt),
            ],
        )
    ).text

5. Define processing function

In [6]:
import json
from datasets import Dataset
from time import sleep
import numpy
from tqdm import tqdm

def translateDataset(dataset: Dataset, sys_prompt: str, name: str):
    
    # Create or load checkpoint
    batchPath = f"{cacheDir}/{name}.npy"
    if (os.path.isfile(batchPath)):
        batchCheckpoint = numpy.load(batchPath)
        print(f"Resuming {name} from batch {batchCheckpoint}")
    else:
        batchCheckpoint = -1
        numpy.save(batchPath, batchCheckpoint)

    for batchNumber, batchContent in enumerate(tqdm(dataset, desc=f"Translating {name}")):

        # Skip if batch already processed
        if (batchNumber <= batchCheckpoint):
            continue

        # Parse and compose input
        input_dict = {s: t for s, t in zip(batchContent['serial'], batchContent['text'])}
        input_keys = set(input_dict.keys())

        success = False;
        retries = 0;
        while not success:
            try:
                input_json = json.dumps(input_dict, indent=4, ensure_ascii=True)
                output_dict = json.loads(generate(input_json, sys_prompt))
                output_keys = set(output_dict.keys())
                if (input_keys == output_keys):
                    success = True
                else:
                    retries += 1
                    tqdm.write(f"[WARN]: Missing data in batch output, retrying {retries}st time")
                    sleep(60)
            except Exception as e:
                retries += 1
                tqdm.write(f"[WARN] Malformed data or API error, retrying {retries}st time")
                if (retries != 0 and retries % 3 == 0):
                    tqdm.write(f"[WARN] Waiting 1h after {retries} errors: {e}")
                    sleep(3600)
                else:
                    sleep(60)

        with open(f"{cacheDir}/{name}.jsonl", "a", encoding="utf-8") as f:
            for i in range(len(batchContent['serial'])):  
                new_row = {
                    "id": batchContent['id'][i],
                    "text": output_dict[batchContent['serial'][i]],
                    "labels": batchContent['labels'][i],
                    "language": "it"
                }
                f.write(json.dumps(new_row, ensure_ascii=False) + "\n")

        batchCheckpoint += 1
        numpy.save(batchPath, batchCheckpoint)

6. Process datasets

In [7]:
system_prompt_italian = """
You are an expert socio-linguist, native Italian translator, and specialist in Italian Web Culture (Reddit, Twitch, TikTok) and Social Media Sentiment/Emotion Analysis.

Your task is to translate a JSON object containing English Reddit comments into natural, colloquial, and highly informal Italian "internet speak". The output is strictly for training an Emotion Analysis model.

### STRICT DIRECTIVES

1. ABSOLUTE EMOTION & TOXICITY PRESERVATION (CRITICAL): 
The primary goal is emotion analysis. You must maintain the EXACT emotional polarity, intensity, and nuance of the original text. 
- DO NOT censor, soften, or sanitize toxic, offensive, misogynistic, or aggressive language. If the original uses highly derogatory terms (e.g., treating people as objects/animals), the Italian translation MUST be equally derogatory and heavy.
- NEVER translate English idioms literally. Find the exact Italian cultural equivalent that carries the same emotional weight (e.g., translate "come out of the closet" as "fare coming out", NOT "uscire dall'armadio").
- Avoid cross-language hallucinations. Ensure the output is strictly Italian (e.g., do not use Portuguese words like "Adorei", use "Adoravo" or "Ho adorato").

2. TONE & TYPOGRAPHY MIRRORING: 
Replicate the visual energy of the text. If the original uses ALL CAPS, keyboard mashing (asdfghjkl), repeated punctuation (?!?!?!!!), intentional typos, or emojis, you MUST replicate them in the Italian output exactly.

3. SLANG & JARGON RULES:
- KEEP Reddit/Web specifics in English: OP, sub, upvote, downvote, karma, mod, thread, AMA, AITA, TL;DR, ban.
- USE modern Italian web loans/slang NATIVELY (e.g., cringe, bro, triggerato, blastare, basato, chad, boomer).
- AVOID unnatural hybrids or forced internet speak. For example, do not write "farmi un lmao" (unnatural); either use "lmao" seamlessly in the sentence or use a native equivalent like "sto morendo".

4. ZERO-TOLERANCE JSON INTEGRITY:
- Output STRICTLY AND EXCLUSIVELY a raw, valid JSON object.
- DO NOT wrap the output in markdown code blocks (Do NOT use ```json ... ```).
- NO preamble, NO greetings, NO post-translation commentary. Just the JSON starting with { and ending with }.
- The output keys MUST perfectly match the input keys in the exact same order.
- CRITICAL: Ensure all internal quotation marks within the translated strings are properly escaped (\") to avoid breaking the JSON structure.
"""

In [8]:
# Italian
translateDataset(dataset_goemotions_test, system_prompt_italian, "it-test")
translateDataset(dataset_goemotions_train, system_prompt_italian, "it-train")
translateDataset(dataset_goemotions_validation, system_prompt_italian, "it-validation")

Resuming it-test from batch 54


Translating it-test: 100%|██████████| 55/55 [00:00<00:00, 2460.50it/s]


Resuming it-train from batch 434


Translating it-train: 100%|██████████| 435/435 [00:00<00:00, 2310.48it/s]


Resuming it-validation from batch 54


Translating it-validation: 100%|██████████| 55/55 [00:00<00:00, 2071.37it/s]


7. Merge, clean and generate final dataset

In [9]:
from datasets import DatasetDict, concatenate_datasets

# Retrive
datasets_generated_it = load_dataset("json", data_files={
    "train": f"{cacheDir}/it-train.jsonl",
    "test": f"{cacheDir}/it-test.jsonl",
    "validation": f"{cacheDir}/it-validation.jsonl"
})
dataset_generated_train_it = datasets_generated_it["train"].map(lambda x: {"id": "IT" + x["id"][2:]})
dataset_generated_test_it = datasets_generated_it["test"].map(lambda x: {"id": "IT" + x["id"][2:]})
dataset_generated_validation_it = datasets_generated_it["validation"].map(lambda x: {"id": "IT" + x["id"][2:]})

# Merge
dataset_final_train: Dataset = concatenate_datasets([dataset_original_train, dataset_generated_train_it]
    ).map(lambda x: {**x, "labels": [int(i) for i in x["labels"].strip("[]").split(",")]})
dataset_final_test: Dataset = concatenate_datasets([dataset_original_test, dataset_generated_test_it]
    ).map(lambda x: {**x, "labels": [int(i) for i in x["labels"].strip("[]").split(",")]})
dataset_final_validation: Dataset = concatenate_datasets([dataset_original_validation, dataset_generated_validation_it]
    ).map(lambda x: {**x, "labels": [int(i) for i in x["labels"].strip("[]").split(",")]})

# Pack & remove empty values
dataset_final =  DatasetDict()
dataset_final["train"] = dataset_final_train.filter(lambda example: example["text"] is not None and len(example["text"].strip()) > 0)
dataset_final["test"] = dataset_final_test.filter(lambda example: example["text"] is not None and len(example["text"].strip()) > 0)
dataset_final["validation"] = dataset_final_validation.filter(lambda example: example["text"] is not None and len(example["text"].strip()) > 0)
dataset_final = dataset_final.shuffle()

# Save
dataset_final.save_to_disk("./data/multilang_goemotions_expanded")

Saving the dataset (0/1 shards):   0%|          | 0/303856 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/37982 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/37968 [00:00<?, ? examples/s]

In [10]:
# Push to HF
from huggingface_hub import notebook_login
notebook_login()
dataset_final.push_to_hub(repo_id="43ntropy/synthlangs-goemotions")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/43ntropy/synthlangs-goemotions/commit/0d6bb673a5e188908e8a1649ebd96bf044283ce2', commit_message='Upload dataset', commit_description='', oid='0d6bb673a5e188908e8a1649ebd96bf044283ce2', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/43ntropy/synthlangs-goemotions', endpoint='https://huggingface.co', repo_type='dataset', repo_id='43ntropy/synthlangs-goemotions'), pr_revision=None, pr_num=None)